# El Niño 2026: Seasonal Outlook for Africa
## Using ECMWF SEAS5 and ERA5 via Copernicus CDS

**Author:** Valters Žeižis, ECMWF  
**Date:** May 2026  
**Context:** Continental Preparedness Framework 2026 El Niño


## What this notebook does

Demonstrates how to use free ECMWF data from the Copernicus Climate Data Store (CDS)
to build a regional seasonal outlook for Africa, directly relevant to the 2026 El Niño
preparedness framework.

**Key question:** What does the current seasonal forecast tell us about precipitation
anomalies across the four SEWA regions for the coming months — and where are the
highest drought and flood risks?

## Why these data sources?

| Source | What it provides | Why it matters |
|---|---|---|
| **SEAS5** | 51-member ensemble, 1–7 months ahead | Primary signal for seasonal drought/flood risk |
| **ERA5 climatology** | Historical baseline 1993–2016 | Required to compute anomalies vs normal |

Both are **free** via [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu).

## The 2026 El Niño context

- **82% probability** of El Niño emerging May–July 2026 (NOAA Climate Prediction Centre, May 2026)
- **96% probability** persisting through northern hemisphere winter 2026-27
- Some models forecast ONI > 3°C — potentially among the strongest on record (WMO, April 2026)

> **Note on uncertainty:** Probability figures cited above are as of May 2026 and should be
> updated as the season evolves. The SEAS5 signal shown throughout this notebook is probabilistic —
> ensemble spread sections (9–11) quantify forecast confidence explicitly.

**Expected Africa impacts by region:**
- **Western Africa**: Reduced monsoon, Sahel drought risk
- **Eastern Africa**: Enhanced short rains (Oct-Dec), elevated flood risk
- **Southern Africa**: Drought in summer rainfall regions, maize belt at risk (Dec-Feb peak)
- **Central Africa**: Mixed signal, slightly wetter during onset


## 1) Setup

In [ ]:
import sys
import cdsapi
import cfgrib
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Rectangle
from pathlib import Path
from datetime import datetime
import calendar as cal_module

# Attribution utility — lives in project root
sys.path.insert(0, str(Path().resolve().parent))
from add_attribution import save_with_attribution

try:
    from _utils import get_data_dir
    DATA_DIR = Path(get_data_dir())
except Exception:
    DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

c = cdsapi.Client(quiet=True)

now = datetime.utcnow()
FCST_YEAR  = str(now.year)
FCST_MONTH = f'{now.month:02d}'

# Unit conversion constants
SECONDS_PER_MONTH = 30 * 24 * 3600   # for SEAS5 tprate (m/s -> mm/month)
DAYS_PER_MONTH    = 30                # for ERA5 tp (m/day -> mm/month)

# SEWA focus regions
REGIONS = {
    'Western Africa (AGRHYMET)':  {'lon': [-20, 20], 'lat': [4,  20],  'color': '#F59E0B'},
    'Central Africa (CAPC-AC)':   {'lon': [8,  30],  'lat': [-5, 10],  'color': '#8B5CF6'},
    'Eastern Africa (ICPAC)':     {'lon': [28, 52],  'lat': [-5, 20],  'color': '#3B82F6'},
    'Southern Africa (SADC-CSC)': {'lon': [10, 50],  'lat': [-35, -5], 'color': '#EF4444'},
}

print(f'Data cache: {DATA_DIR}')
print(f'Forecast initialisation: {FCST_YEAR}-{FCST_MONTH}')
print(f'CDS client: ready')


## 2) Fetch SEAS5: Seasonal Precipitation Forecast

**What:** Monthly mean total precipitation for Africa, 51 ensemble members, lead months 1–6.

**Why 6 months:** A May initialisation gives skill out to October-November, covering:
- June-August: West African monsoon (drought signal emerges early)
- September-November: Eastern Africa short rains (flood signal peaks)
- October-December: Southern Africa summer onset (drought signal peaks)

**Why 51 members:** Ensemble spread quantifies forecast confidence.
Narrow spread = strong signal. Wide spread = high uncertainty.
Sections 9–11 use the full per-member data to show this explicitly.

> ⚠️ Accept licence at [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu/datasets/seasonal-monthly-single-levels?tab=download#manage-licences)


In [ ]:
seas5_path = DATA_DIR / f'seas5_tp_africa_{FCST_YEAR}{FCST_MONTH}.grib'

if not seas5_path.exists():
    print('Downloading SEAS5... (~2-3 min first run)')
    c.retrieve(
        'seasonal-monthly-single-levels',
        {
            'originating_centre': 'ecmwf',
            'system': '51',
            'variable': 'total_precipitation',
            'product_type': 'monthly_mean',
            'year': FCST_YEAR,
            'month': FCST_MONTH,
            'leadtime_month': ['1', '2', '3', '4', '5', '6'],
            'area': [40, -20, -40, 55],
            'data_format': 'grib',
        },
        str(seas5_path)
    )
    print(f'Saved: {seas5_path.name} ({seas5_path.stat().st_size/1e6:.1f} MB)')
else:
    print(f'Cached: {seas5_path.name} ({seas5_path.stat().st_size/1e6:.1f} MB)')


## 3) Fetch ERA5 Climatology: 1993–2016

**What:** Monthly mean precipitation from ERA5, 1993–2016, all 12 calendar months.

**Why 1993–2016:** Standard SEAS5 hindcast period used by C3S for skill assessment.
Using the same reference ensures anomaly calculation is methodologically consistent
with published SEAS5 verification.

> ⚠️ Accept licence at [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means?tab=download#manage-licences)


In [ ]:
era5_clim_path = DATA_DIR / 'era5_tp_africa_clim_1993_2016.grib'

if not era5_clim_path.exists():
    print('Downloading ERA5 climatology 1993-2016... (~3-5 min)')
    c.retrieve(
        'reanalysis-era5-single-levels-monthly-means',
        {
            'product_type': 'monthly_averaged_reanalysis',
            'variable': 'total_precipitation',
            'year': [str(y) for y in range(1993, 2017)],
            'month': [f'{m:02d}' for m in range(1, 13)],
            'time': '00:00',
            'area': [40, -20, -40, 55],
            'data_format': 'grib',
        },
        str(era5_clim_path)
    )
    print(f'Saved: {era5_clim_path.name} ({era5_clim_path.stat().st_size/1e6:.1f} MB)')
else:
    print(f'Cached: {era5_clim_path.name} ({era5_clim_path.stat().st_size/1e6:.1f} MB)')


## 4) Load Data and Compute Anomalies

**Unit note:**
- SEAS5 `tprate`: precipitation **rate** in m/s → convert: × seconds_per_month × 1000 → mm/month
- ERA5 `tp`: precipitation **rate** in m/day → convert: × days_per_month × 1000 → mm/month

**Anomaly method:** For each SEAS5 lead month, extract the matching calendar month
from ERA5 (e.g. lead month 1 from May = June → compare against ERA5 June 1993-2016 mean).

**Key change vs earlier versions:** The per-member array (`fc_tp_members`) is retained
alongside the ensemble mean. This is required for the spread and probability calculations
in Sections 9–11.


In [ ]:
# Load SEAS5
ds_fc_list = cfgrib.open_datasets(str(seas5_path))
ds_fc = ds_fc_list[0]
tp_vars = [v for v in ds_fc.data_vars if 'tp' in v.lower() or 'precip' in v.lower()]
tp_key = tp_vars[0] if tp_vars else list(ds_fc.data_vars)[0]
fc_tp = ds_fc[tp_key]
print(f'SEAS5: variable={tp_key}, units={fc_tp.attrs.get("units","?")}, shape={fc_tp.shape}')
print(f'SEAS5 dims: {list(fc_tp.dims)}')

# Identify member and step dimensions
member_dims = [d for d in fc_tp.dims if 'number' in d or 'member' in d]
step_dims   = [d for d in fc_tp.dims if d not in ('latitude', 'longitude') and d not in member_dims]
print(f'Member dim(s): {member_dims}  |  Step dim(s): {step_dims}')

# Load ERA5
ds_clim_list = cfgrib.open_datasets(str(era5_clim_path))
ds_clim = ds_clim_list[0]
clim_vars = [v for v in ds_clim.data_vars if 'tp' in v.lower() or 'precip' in v.lower()]
clim_key = clim_vars[0] if clim_vars else list(ds_clim.data_vars)[0]
cl_tp = ds_clim[clim_key]
print(f'\nERA5: variable={clim_key}, units={cl_tp.attrs.get("units","?")}, shape={cl_tp.shape}')


In [ ]:
# Ensemble mean across members (keep step dimension)
fc_mean_all = fc_tp.mean(dim=member_dims, keep_attrs=True) * SECONDS_PER_MONTH * 1000
print(f'SEAS5 ensemble mean: {float(fc_mean_all.min()):.1f} to {float(fc_mean_all.max()):.1f} mm/month')

# ERA5 time index
era5_times = pd.DatetimeIndex(cl_tp.time.values)

# Lead month calendar mapping
INIT_MONTH  = int(FCST_MONTH)
lead_months = list(range(1, 7))
valid_months = [(INIT_MONTH + l - 1) % 12 + 1 for l in lead_months]

print(f'\nInitialisation: {FCST_YEAR}-{FCST_MONTH}')
print(f'Lead months 1-6 → calendar months: {valid_months}')

# Build lead_results — store both ensemble mean AND per-member anomalies
lead_results = {}
for lead_idx, (lead, cal_month) in enumerate(zip(lead_months, valid_months)):

    # --- ensemble mean for this lead ---
    if step_dims:
        fc_lead = fc_mean_all.isel({step_dims[0]: lead_idx}, drop=True)
    else:
        fc_lead = fc_mean_all

    # --- per-member array for this lead (retained for spread/probability) ---
    fc_members_lead = fc_tp.isel({step_dims[0]: lead_idx}, drop=True) * SECONDS_PER_MONTH * 1000
    # fc_members_lead has dims (number/member, latitude, longitude)

    # --- ERA5 climatology for matching calendar month ---
    month_mask = era5_times.month == cal_month
    cl_month   = cl_tp.isel(time=month_mask)
    clim       = cl_month.mean(dim='time', keep_attrs=True) * DAYS_PER_MONTH * 1000

    # Sort grids consistently
    clim_s = clim.sortby('latitude').sortby('longitude')
    fc_s   = fc_lead.sortby('latitude').sortby('longitude')
    fc_mem_s = fc_members_lead.sortby('latitude').sortby('longitude')

    # Regrid ERA5 → SEAS5 grid
    clim_i = clim_s.interp(
        latitude=fc_s.latitude,
        longitude=fc_s.longitude,
        method='linear'
    )

    # Ensemble mean anomaly
    anom = fc_s - clim_i

    # Per-member anomaly (broadcast clim_i across member dim)
    anom_members = fc_mem_s - clim_i

    lead_results[lead] = {
        'fc':          fc_s,
        'clim':        clim_i,
        'anom':        anom,
        'anom_members': anom_members,   # shape: (n_members, lat, lon)
        'cal_month':   cal_month,
        'label':       pd.Timestamp(f'2026-{cal_month:02d}-01').strftime('%B'),
    }
    n_mem = anom_members.shape[0] if anom_members.ndim > 2 else 1
    print(f'  Lead +{lead} ({lead_results[lead]["label"]}): '
          f'mean_anom={float(anom.mean()):+.1f} mm/month  '
          f'members={n_mem}')


## 5) Regional Signal — All Lead Months

Area-averaged ensemble mean anomaly per SEWA region × lead month.


In [ ]:
print("SEAS5 El Niño 2026 Signal by Region and Lead Month")
print("=" * 80)
header = f"{'Region':<28}" + "".join([f"  {lead_results[l]['label'][:3]:>6}" for l in range(1,7)])
print(header)
print("-" * 80)

for name, reg in REGIONS.items():
    row = f"{name:<28}"
    for lead in range(1, 7):
        res  = lead_results[lead]
        anom = res['anom']
        lat_mask = (anom.latitude >= reg['lat'][0]) & (anom.latitude <= reg['lat'][1])
        lon_mask = (anom.longitude >= reg['lon'][0]) & (anom.longitude <= reg['lon'][1])
        anom_reg = float(anom.where(lat_mask & lon_mask).mean())
        sig = "🔴" if anom_reg < -15 else ("🔵" if anom_reg > 15 else "⚪")
        row += f"  {anom_reg:>+5.0f}{sig}"
    print(row)

print()
print("Values = ensemble mean anomaly vs ERA5 1993-2016 (mm/month)")
print("🔴 < -15mm  drier   🔵 > +15mm  wetter   ⚪ near-normal")
print("Note: ensemble mean only — see Sections 9-11 for confidence/spread")


## 6) Maps: Seasonal Precipitation Anomaly

Ensemble mean anomaly for Lead +1 (June) and Lead +4 (September).


In [ ]:
def plot_anomaly_map(ax, anom, title, vmin=-80, vmax=80):
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
    ax.add_feature(cfeature.LAND, facecolor='#F5F5F5', alpha=0.2)
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)

    norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
    im = ax.contourf(
        anom.longitude, anom.latitude, anom,
        levels=np.linspace(vmin, vmax, 21),
        cmap='RdBu', norm=norm,
        transform=ccrs.PlateCarree(), extend='both'
    )
    plt.colorbar(im, ax=ax, label='mm/month', orientation='horizontal',
                 pad=0.05, fraction=0.04)

    for rname, reg in REGIONS.items():
        rect = Rectangle(
            (reg['lon'][0], reg['lat'][0]),
            reg['lon'][1] - reg['lon'][0],
            reg['lat'][1] - reg['lat'][0],
            linewidth=2, edgecolor=reg['color'],
            facecolor='none', transform=ccrs.PlateCarree()
        )
        ax.add_patch(rect)
        ax.text(reg['lon'][0] + 0.5, reg['lat'][1] - 2,
                rname.split('(')[0].strip(),
                transform=ccrs.PlateCarree(),
                fontsize=7, color=reg['color'], fontweight='bold')

    ax.set_extent([-20, 55, -40, 40])
    ax.set_title(title, fontsize=10)
    return im

fig, axes = plt.subplots(1, 2, figsize=(16, 7),
    subplot_kw={'projection': ccrs.PlateCarree()})

fig.suptitle(
    f'SEAS5 Seasonal Precipitation Anomaly vs ERA5 1993–2016\n'
    f'Initialised: {FCST_YEAR}-{FCST_MONTH}  |  '
    f'El Niño 2026: 82% onset probability May–Jul (NOAA CPC, May 2026)',
    fontsize=11, fontweight='bold', y=1.01
)

for ax, lead in zip(axes, [1, 4]):
    res = lead_results[lead]
    plot_anomaly_map(ax, res['anom'],
        f"Lead +{lead}: {res['label']}\n(blue = wetter, red = drier than normal)")

plt.tight_layout()
save_with_attribution(fig, DATA_DIR / 'el_nino_2026_seasonal_outlook.png', dpi=150)
plt.show()


## 7) Multi-Lead Timeline: Evolving El Niño Signal

Six-panel figure showing ensemble mean anomaly evolving June through November.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10),
    subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

fig.suptitle(
    f'SEAS5 El Niño 2026: Precipitation Anomaly Evolution (Lead +1 to +6)\n'
    f'Initialised {FCST_YEAR}-{FCST_MONTH}  |  vs ERA5 climatology 1993–2016',
    fontsize=12, fontweight='bold'
)

for ax, lead in zip(axes, range(1, 7)):
    res = lead_results[lead]
    plot_anomaly_map(ax, res['anom'],
        f"Lead +{lead}: {res['label']} 2026",
        vmin=-80, vmax=80)

plt.tight_layout()
save_with_attribution(fig, DATA_DIR / 'el_nino_2026_full_timeline.png', dpi=150)
plt.show()


## 8) The Anticipatory Action Chain

For early warning systems, seasonal forecasts are the **trigger layer** —
they initiate preparedness actions months before an event:

```
SEAS5 (1–7 months ahead)
    → Is this a drought / flood season?
    → Pre-position resources, activate coordination mechanisms
    → Inform the Continental Preparedness Framework

S2S sub-seasonal (2–6 weeks ahead)
    → Which weeks are anomalous?
    → Activate national NMHS protocols, issue advisories

IFS / AIFS medium-range (0–15 days)
    → When exactly? How intense?
    → Issue official warnings, trigger early action

AMSAF nowcasting (0–6 hours)
    → Where right now?
    → Emergency response, evacuations
```

**SEWA context:** The IbF pilot grants (Call opens June 2026) will develop tools
that operationalise exactly this seasonal signal into actionable early warnings
at national NMHS level — using the same SEAS5 data shown here.

**Data access — all free today:**
- SEAS5 + ERA5: [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu)
- IFS/AIFS open data: [data.ecmwf.int](https://data.ecmwf.int) — no account needed


---

## 9) Ensemble Spread — Probability of Below-Normal Precipitation

**What this adds:** The ensemble mean (Sections 5–7) shows *where* the signal points,
but not *how confident* the forecast is. A −30 mm/month mean anomaly means something
very different if 50 of 51 members agree vs if only 26 do.

**Method:** For each grid point and each lead month, count the fraction of the 51
ensemble members showing anomaly below −15 mm/month (meaningful drought threshold).
Values above 60% indicate a robust signal; values near 33% indicate no skill above
climatological base rate.

> **Interpretation guide:**
> - **> 60%** — strong consensus, suitable for preparedness trigger
> - **40–60%** — moderate signal, monitor closely
> - **< 40%** — near-climatological uncertainty, low confidence


In [ ]:
DROUGHT_THRESHOLD = -15  # mm/month

fig, axes = plt.subplots(2, 3, figsize=(18, 10),
    subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

fig.suptitle(
    f'SEAS5 El Niño 2026: Probability of Below-Normal Precipitation (< −15 mm/month)\n'
    f'Initialised {FCST_YEAR}-{FCST_MONTH}  |  51-member ensemble',
    fontsize=12, fontweight='bold'
)

prob_cmap = plt.cm.YlOrRd
prob_levels = np.linspace(0, 100, 21)

for ax, lead in zip(axes, range(1, 7)):
    res = lead_results[lead]
    am  = res['anom_members']   # (n_members, lat, lon)

    # Fraction of members below threshold, expressed as %
    prob_drought = (am < DROUGHT_THRESHOLD).mean(dim=am.dims[0]) * 100

    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, linestyle=':')
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.4)

    im = ax.contourf(
        prob_drought.longitude, prob_drought.latitude, prob_drought,
        levels=prob_levels, cmap=prob_cmap,
        transform=ccrs.PlateCarree(), extend='neither'
    )
    plt.colorbar(im, ax=ax, label='% members', orientation='horizontal',
                 pad=0.05, fraction=0.04)

    # Add 60% contour as a bold signal line
    ax.contour(
        prob_drought.longitude, prob_drought.latitude, prob_drought,
        levels=[60], colors='black', linewidths=1.5,
        transform=ccrs.PlateCarree()
    )

    # Region boxes
    for rname, reg in REGIONS.items():
        rect = Rectangle(
            (reg['lon'][0], reg['lat'][0]),
            reg['lon'][1] - reg['lon'][0],
            reg['lat'][1] - reg['lat'][0],
            linewidth=2, edgecolor=reg['color'],
            facecolor='none', transform=ccrs.PlateCarree()
        )
        ax.add_patch(rect)

    ax.set_extent([-20, 55, -40, 40])
    ax.set_title(f"Lead +{lead}: {res['label']} 2026\n(black contour = 60% threshold)", fontsize=10)

plt.tight_layout()
save_with_attribution(fig, DATA_DIR / 'el_nino_2026_prob_drought.png', dpi=150)
plt.show()


## 10) Tercile Probabilities by Region

**What this adds:** WMO standard seasonal forecast format. For each SEWA region
and each lead month, the 51 ensemble members are classified into:
- **Below normal** (BN): bottom tercile of ERA5 1993–2016 climatology
- **Near normal** (NN): middle tercile
- **Above normal** (AN): top tercile

A climatological (no-skill) forecast gives 33/33/33%. Departure from this baseline
shows where the forecast has useful signal.

**How tercile boundaries are derived:** From the ERA5 1993–2016 distribution for
each region × calendar month, using the 33rd and 67th percentiles of annual values.


In [ ]:
# Compute ERA5 tercile boundaries per region × calendar month
# Uses the 24 annual values (1993-2016) for each region/month

era5_times = pd.DatetimeIndex(cl_tp.time.values)

def region_era5_mean(cl_tp, reg):
    """Area-average ERA5 over a region box, return monthly time series in mm/month."""
    lat_mask = (cl_tp.latitude >= reg['lat'][0]) & (cl_tp.latitude <= reg['lat'][1])
    lon_mask = (cl_tp.longitude >= reg['lon'][0]) & (cl_tp.longitude <= reg['lon'][1])
    return float(cl_tp.where(lat_mask & lon_mask).mean(['latitude', 'longitude'])) * DAYS_PER_MONTH * 1000

# Build ERA5 regional climatology: dict[region_name][cal_month] -> array of 24 values
era5_regional_clim = {}
for rname, reg in REGIONS.items():
    era5_regional_clim[rname] = {}
    for m in range(1, 13):
        mask = era5_times.month == m
        vals = []
        for i, t in enumerate(cl_tp.time.values):
            if pd.Timestamp(t).month == m:
                lat_m = (cl_tp.latitude >= reg['lat'][0]) & (cl_tp.latitude <= reg['lat'][1])
                lon_m = (cl_tp.longitude >= reg['lon'][0]) & (cl_tp.longitude <= reg['lon'][1])
                v = float(cl_tp.isel(time=i).where(lat_m & lon_m).mean()) * DAYS_PER_MONTH * 1000
                vals.append(v)
        era5_regional_clim[rname][m] = np.array(vals)

print("ERA5 regional climatology computed.")
print(f"Example — Southern Africa, October: mean={np.mean(era5_regional_clim['Southern Africa (SADC-CSC)'][10]):.1f} mm/month")


In [ ]:
# Compute tercile probabilities for each region × lead month
# For each lead, count members in BN / NN / AN terciles

tercile_results = {}  # [region_name][lead] -> {'BN': %, 'NN': %, 'AN': %}

for rname, reg in REGIONS.items():
    tercile_results[rname] = {}
    for lead in range(1, 7):
        res = lead_results[lead]
        cal_month = res['cal_month']

        # ERA5 tercile boundaries for this region/month
        era5_vals = era5_regional_clim[rname][cal_month]
        t33 = np.percentile(era5_vals, 33.3)
        t67 = np.percentile(era5_vals, 66.7)

        # Per-member area-average anomaly + climatology → absolute precipitation
        am = res['anom_members']
        clim_val = float(res['clim'].where(
            (res['clim'].latitude >= reg['lat'][0]) &
            (res['clim'].latitude <= reg['lat'][1]) &
            (res['clim'].longitude >= reg['lon'][0]) &
            (res['clim'].longitude <= reg['lon'][1])
        ).mean())

        member_vals = []
        for mi in range(am.shape[0]):
            lat_m = (am.latitude >= reg['lat'][0]) & (am.latitude <= reg['lat'][1])
            lon_m = (am.longitude >= reg['lon'][0]) & (am.longitude <= reg['lon'][1])
            mv = float(am.isel({am.dims[0]: mi}).where(lat_m & lon_m).mean()) + clim_val
            member_vals.append(mv)
        member_vals = np.array(member_vals)

        n = len(member_vals)
        pct_bn = np.sum(member_vals < t33) / n * 100
        pct_an = np.sum(member_vals > t67) / n * 100
        pct_nn = 100 - pct_bn - pct_an

        tercile_results[rname][lead] = {'BN': pct_bn, 'NN': pct_nn, 'AN': pct_an}

print("Tercile probabilities computed.")
print(f"\nSample — Southern Africa:")
for lead in range(1, 7):
    t = tercile_results['Southern Africa (SADC-CSC)'][lead]
    print(f"  Lead +{lead} ({lead_results[lead]['label'][:3]}): "
          f"BN={t['BN']:.0f}%  NN={t['NN']:.0f}%  AN={t['AN']:.0f}%")


In [ ]:
# Plot tercile probability stacked bars — one panel per region
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

month_labels = [lead_results[l]['label'][:3] for l in range(1, 7)]
x = np.arange(6)
width = 0.6

BN_COLOR = '#D97706'   # amber — below normal / drier
NN_COLOR = '#E5E7EB'   # light grey — near normal
AN_COLOR = '#2563EB'   # blue — above normal / wetter

for ax, (rname, reg) in zip(axes, REGIONS.items()):
    bn_vals = [tercile_results[rname][l]['BN'] for l in range(1, 7)]
    nn_vals = [tercile_results[rname][l]['NN'] for l in range(1, 7)]
    an_vals = [tercile_results[rname][l]['AN'] for l in range(1, 7)]

    bars_bn = ax.bar(x, bn_vals, width, label='Below normal', color=BN_COLOR)
    bars_nn = ax.bar(x, nn_vals, width, bottom=bn_vals, label='Near normal', color=NN_COLOR)
    bars_an = ax.bar(x, an_vals, width,
                     bottom=[b + n for b, n in zip(bn_vals, nn_vals)],
                     label='Above normal', color=AN_COLOR)

    # Climatological reference line
    ax.axhline(33.3, color='black', linewidth=1.0, linestyle='--', alpha=0.5, label='Climatology (33%)')

    ax.set_xticks(x)
    ax.set_xticklabels(month_labels)
    ax.set_ylim(0, 100)
    ax.set_ylabel('Probability (%)')
    ax.set_title(rname, color=reg['color'], fontweight='bold', fontsize=11)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlabel(f'Lead month (Initialised {FCST_YEAR}-{FCST_MONTH})')

    # Annotate BN values
    for i, v in enumerate(bn_vals):
        if v > 10:
            ax.text(i, v / 2, f'{v:.0f}%', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

fig.suptitle(
    f'SEAS5 El Niño 2026: Tercile Probabilities by SEWA Region\n'
    f'Initialised {FCST_YEAR}-{FCST_MONTH}  |  51-member ensemble  |  ERA5 1993–2016 baseline',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
save_with_attribution(fig, DATA_DIR / 'el_nino_2026_tercile_probs.png', dpi=150)
plt.show()


## 11) Ensemble Plume — Member Agreement Over Time

**What this adds:** Shows *all 51 members* as individual lines (spaghetti),
with the ensemble mean overlaid. This is the clearest way to see:

- **Tight cluster** → high confidence, strong signal
- **Wide spread** → model uncertainty is large, signal weaker
- **Bimodal distribution** → some members in opposite regime (inspect carefully)

Each panel is one SEWA region. The x-axis runs June through November 2026.
The y-axis is area-averaged precipitation anomaly in mm/month.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, (rname, reg) in zip(axes, REGIONS.items()):
    member_series = []  # list of arrays, one per member, length = 6 leads

    for lead in range(1, 7):
        res = lead_results[lead]
        am  = res['anom_members']
        lat_m = (am.latitude >= reg['lat'][0]) & (am.latitude <= reg['lat'][1])
        lon_m = (am.longitude >= reg['lon'][0]) & (am.longitude <= reg['lon'][1])

        lead_member_vals = []
        for mi in range(am.shape[0]):
            v = float(am.isel({am.dims[0]: mi}).where(lat_m & lon_m).mean())
            lead_member_vals.append(v)
        member_series.append(lead_member_vals)

    # member_series: shape (6 leads, n_members) -> transpose to (n_members, 6)
    member_array = np.array(member_series).T  # (n_members, 6)
    ensemble_mean = member_array.mean(axis=0)

    x = np.arange(6)
    month_labels = [lead_results[l]['label'][:3] for l in range(1, 7)]

    # Spaghetti: each member as thin translucent line
    for mi in range(member_array.shape[0]):
        ax.plot(x, member_array[mi], color=reg['color'], alpha=0.15, linewidth=0.8)

    # Interquartile range shading
    q25 = np.percentile(member_array, 25, axis=0)
    q75 = np.percentile(member_array, 75, axis=0)
    ax.fill_between(x, q25, q75, color=reg['color'], alpha=0.25, label='IQR (25–75%)')

    # Ensemble mean
    ax.plot(x, ensemble_mean, color=reg['color'], linewidth=2.5,
            marker='o', markersize=5, label='Ensemble mean', zorder=5)

    # Zero reference
    ax.axhline(0, color='black', linewidth=1.0, linestyle='--', alpha=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(month_labels)
    ax.set_ylabel('Anomaly (mm/month)')
    ax.set_title(rname, color=reg['color'], fontweight='bold', fontsize=11)
    ax.legend(fontsize=9)
    ax.set_xlabel(f'Lead month (Initialised {FCST_YEAR}-{FCST_MONTH})')

    # Annotate spread at final lead
    spread = member_array[:, -1].std()
    ax.annotate(f'σ={spread:.0f} mm', xy=(5, ensemble_mean[-1]),
                xytext=(4.3, ensemble_mean[-1] + 8),
                fontsize=8, color=reg['color'],
                arrowprops=dict(arrowstyle='->', color=reg['color'], lw=1))

fig.suptitle(
    f'SEAS5 El Niño 2026: Ensemble Plume by SEWA Region\n'
    f'Initialised {FCST_YEAR}-{FCST_MONTH}  |  51 members  |  Anomaly vs ERA5 1993–2016',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
save_with_attribution(fig, DATA_DIR / 'el_nino_2026_ensemble_plume.png', dpi=150)
plt.show()


## 12) Confidence Summary Table

Combines the ensemble mean signal with spread information into a single
decision-support table. Intended for use in the preparedness framework briefing.

**Confidence scoring:**
- Signal strength: |mean anomaly| relative to threshold (15 mm/month)
- Spread: standard deviation of member area-averages
- Signal-to-noise ratio (SNR): |mean| / std — higher = more confident

> **Preparedness note:** Cells marked 🔴 HIGH with SNR > 1.5 are the most
> suitable candidates for defining anticipatory action triggers from SEAS5.


In [ ]:
print("SEAS5 El Niño 2026 — Signal + Confidence Summary")
print("=" * 110)
header = f"{'Region':<28}  {'Lead':<6}  {'Month':<10}  {'Mean (mm)':<11}  {'Spread σ':<10}  {'SNR':<6}  {'Signal':<10}  {'Confidence'}"
print(header)
print("-" * 110)

for rname, reg in REGIONS.items():
    for lead in range(1, 7):
        res  = lead_results[lead]
        am   = res['anom_members']
        lat_m = (am.latitude >= reg['lat'][0]) & (am.latitude <= reg['lat'][1])
        lon_m = (am.longitude >= reg['lon'][0]) & (am.longitude <= reg['lon'][1])

        member_vals = np.array([
            float(am.isel({am.dims[0]: mi}).where(lat_m & lon_m).mean())
            for mi in range(am.shape[0])
        ])
        mean_anom = member_vals.mean()
        spread    = member_vals.std()
        snr       = abs(mean_anom) / spread if spread > 0 else 0

        if mean_anom < -15:
            signal = "🔴 DRIER"
        elif mean_anom > 15:
            signal = "🔵 WETTER"
        else:
            signal = "⚪ NEUTRAL"

        if snr > 1.5:
            conf = "HIGH"
        elif snr > 0.8:
            conf = "MODERATE"
        else:
            conf = "LOW"

        print(f"{rname:<28}  +{lead:<5}  {res['label']:<10}  {mean_anom:>+8.1f}    "
              f"{spread:>7.1f}    {snr:>5.2f}  {signal:<12}  {conf}")
    print()

print()
print("SNR = |mean anomaly| / ensemble std dev")
print("HIGH confidence = SNR > 1.5  |  MODERATE = 0.8-1.5  |  LOW = < 0.8")


---

## Key Messages

### Signal summary (SEAS5, May 2026 initialisation — ensemble mean)
- **Western Africa**: Drier than normal signal emerging through monsoon season → Sahel drought watch
- **Eastern Africa**: Near-normal June, wetter signal building toward short rains (Oct-Dec) → flood preparedness
- **Southern Africa**: Drought signal strengthens toward austral summer (Nov-Feb) → maize belt risk
- **Central Africa**: Mixed, slightly wetter — monitor for flash flood and thunderstorm risk

### Confidence (from Sections 9–11)
- **Southern Africa from September onward** is where signal strength and ensemble agreement are both highest — strongest candidate for preparedness triggers
- **Eastern Africa flood signal (Oct-Nov)** is real but ensemble spread is wider — monitor closely
- **Western Africa** signal is weaker and lower confidence throughout — consistent with known lower SEAS5 skill in the Sahel

### For the Continental Preparedness Framework
1. **Trigger thresholds can be defined now** from SEAS5 ensemble probabilities (Section 9–10)
2. **6-month lead time exists** — this is the window for anticipatory action and resource pre-positioning
3. **Data is free and available today** — no procurement barriers for NMHSs
4. **SEWA builds the tools** that turn this seasonal signal into impact-based warnings

*Data: ECMWF SEAS5 via Copernicus CDS · ERA5 via Copernicus CDS*  
*NOAA CPC El Niño outlook: May 2026 · WMO Global Seasonal Climate Update: April 2026*  
*Contact: valters.zeizis@ecmwf.int · support.ecmwf.int*
